# Semana 1 · SQL + Python + PySpark

**Temas:** SELECT · WHERE · ORDER BY · GROUP BY · HAVING · funciones de agregación
**Python:** tipos de datos · listas · diccionarios · funciones · comprensiones

**Datos:** tablas de ejemplo que trae Databricks, `samples.tpch.orders` (pedidos) y `samples.tpch.customer` (clientes).

**Cómo usarlo**
1. Conecta el notebook al cómputo **Serverless** (botón *Connect*, arriba a la derecha).
2. Cada día haz su bloque: primero el **ejemplo**, después los **ejercicios**.
3. Al terminar el día: guarda, haz **commit & push** y anota en una línea qué aprendiste (sirve para tu publicación).

## 0. El puente: la misma consulta en SQL, PySpark y Python

En Databricks, SQL y PySpark son **dos formas de hablar con el mismo motor (Spark)**. Python puro corre solo en tu máquina, y sirve para datos pequeños.

| Quieres... | SQL | PySpark (DataFrame) | Python puro |
|---|---|---|---|
| Elegir columnas | `SELECT a, b` | `df.select("a", "b")` | `[{"a": r["a"]} for r in filas]` |
| Filtrar | `WHERE x > 10` | `df.filter(F.col("x") > 10)` | `[r for r in filas if r["x"] > 10]` |
| Ordenar | `ORDER BY x DESC` | `df.orderBy(F.col("x").desc())` | `sorted(filas, key=lambda r: r["x"], reverse=True)` |
| Agrupar | `GROUP BY k` + `COUNT(*)` | `df.groupBy("k").count()` | un `dict` que va contando |
| Filtrar grupos | `HAVING COUNT(*) > 5` | `.filter(F.col("count") > 5)` después del `groupBy` | `{k: v for k, v in d.items() if v > 5}` |

**Puentes útiles**
- `spark.sql("SELECT ...")` ejecuta SQL desde Python y te devuelve un DataFrame.
- `df.createOrReplaceTempView("mi_vista")` te deja consultar un DataFrame con SQL.
- Después de una celda `%sql`, su resultado queda disponible en Python como `_sqldf`.
- `df.limit(100).collect()` trae filas a Python como una lista de `Row`. `row.asDict()` las convierte en diccionarios. **Úsalo solo con pocas filas.**

In [0]:
from pyspark.sql import functions as F

# 1) SQL desde Python
df_sql = spark.sql("""
    SELECT o_orderstatus, COUNT(*) AS num_pedidos
    FROM samples.tpch.orders
    GROUP BY o_orderstatus
""")
display(df_sql)

# 2) Lo mismo con la API de DataFrames
orders = spark.table("samples.tpch.orders")
display(orders.groupBy("o_orderstatus").agg(F.count("*").alias("num_pedidos")))

# 3) De Spark a Python puro (pocas filas)
filas = [r.asDict() for r in orders.limit(5).collect()]
print(type(filas), type(filas[0]))
print(filas[0])

## Conoce los datos

**orders** (pedidos): `o_orderkey` (id), `o_custkey` (id del cliente), `o_orderstatus` (F = cumplido, O = abierto, P = parcial), `o_totalprice`, `o_orderdate`, `o_orderpriority` ('1-URGENT' ... '5-LOW'), `o_clerk`, `o_shippriority`, `o_comment`

**customer** (clientes): `c_custkey`, `c_name`, `c_address`, `c_nationkey`, `c_phone`, `c_acctbal` (saldo), `c_mktsegment` (segmento), `c_comment`

Primero mira la estructura de cada tabla:

In [0]:
%sql
DESCRIBE samples.tpch.orders

In [0]:
%sql
DESCRIBE samples.tpch.customer

# Parte A · SQL (bloque de 3 h)
Escribe cada respuesta en la celda `%sql` que está debajo del enunciado.

## Día 1 · SELECT, alias, DISTINCT, columnas calculadas
**Ejemplo resuelto:**

In [0]:
%sql
SELECT o_orderkey AS id_pedido, o_totalprice AS total
FROM samples.tpch.orders
LIMIT 5

**Ejercicio 1.** Muestra las primeras 10 filas completas de `orders`.

In [0]:
%sql
SELECT * FROM samples.tpch.orders
LIMIT 10

**Ejercicio 2.** Muestra solo `o_orderkey`, `o_orderdate` y `o_totalprice` de 20 pedidos.

In [0]:
%sql
SELECT o_orderkey AS, o_orderdate, o_totalprice
FROM samples.tpch.orders
LIMIT 20

**Ejercicio 3.** Repite el ejercicio 2, pero renombra las columnas como `id_pedido`, `fecha` y `total`.

In [0]:
%sql
SELECT o_orderkey AS id_pedido, o_orderdate AS fecha, o_totalprice AS total
FROM samples.tpch.orders
LIMIT 20

**Ejercicio 4.** ¿Qué valores distintos tiene `o_orderstatus`?

In [0]:
%sql
SELECT DISTINCT o_orderstatus FROM samples.tpch.orders

**Ejercicio 5.** ¿Qué segmentos de mercado (`c_mktsegment`) existen en `customer`?

In [0]:
%sql
SELECT DISTINCT c_mktsegment FROM samples.tpch.customer

**Ejercicio 6.** Muestra 10 pedidos con su `o_totalprice` y una columna nueva `total_con_iva` (precio × 1.15), redondeada a 2 decimales con `ROUND`.

In [0]:
%sql
SELECT o_orderkey AS idPedido, o_totalprice AS total, round(o_totalprice * 1.15) AS total_con_iva FROM samples.tpch.orders

## Día 2 · WHERE: =, >, AND/OR, IN, BETWEEN, LIKE
**Ejemplo resuelto:**

In [0]:
%sql
SELECT o_orderkey, o_orderstatus, o_totalprice
FROM samples.tpch.orders
WHERE o_orderstatus = 'O' AND o_totalprice > 300000
LIMIT 10

**Ejercicio 7.** Pedidos con `o_totalprice` mayor a 450000.

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 8.** Pedidos con estado `'F'` **y** prioridad `'1-URGENT'` (muestra 20).

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 9.** Pedidos hechos durante 1995 (usa `BETWEEN` con fechas `'1995-01-01'` y `'1995-12-31'`).

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 10.** Clientes de los segmentos `'BUILDING'` o `'FURNITURE'` (usa `IN`).

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 11.** Clientes con saldo negativo (`c_acctbal < 0`).

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 12.** Clientes cuyo nombre termina en `999` (usa `LIKE '%999'`).

In [0]:
%sql
-- tu consulta aquí

## Día 3 · ORDER BY
**Ejemplo resuelto:**

In [0]:
%sql
SELECT o_orderkey, o_orderdate, o_totalprice
FROM samples.tpch.orders
ORDER BY o_orderdate ASC
LIMIT 10

**Ejercicio 13.** Los 10 pedidos más caros.

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 14.** Los 10 clientes con mayor saldo del segmento `'AUTOMOBILE'`.

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 15.** 20 pedidos ordenados por fecha (más antigua primero) y, dentro de la misma fecha, por precio de mayor a menor.

In [0]:
%sql
-- tu consulta aquí

## Día 4 · Funciones de agregación: COUNT, SUM, AVG, MIN, MAX
**Ejemplo resuelto:**

In [0]:
%sql
SELECT COUNT(*) AS total_clientes, ROUND(AVG(c_acctbal), 2) AS saldo_promedio
FROM samples.tpch.customer

**Ejercicio 16.** ¿Cuántos pedidos hay en total?

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 17.** Suma, promedio, mínimo y máximo de `o_totalprice` en una sola consulta (redondea el promedio).

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 18.** ¿Cuántos clientes **distintos** han hecho pedidos? (`COUNT(DISTINCT ...)`)

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 19.** Fecha del primer y del último pedido.

In [0]:
%sql
-- tu consulta aquí

## Día 5 · GROUP BY
**Ejemplo resuelto:**

In [0]:
%sql
SELECT c_mktsegment, COUNT(*) AS num_clientes
FROM samples.tpch.customer
GROUP BY c_mktsegment

**Ejercicio 20.** Número de pedidos por estado (`o_orderstatus`).

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 21.** Total vendido por prioridad, ordenado de mayor a menor.

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 22.** Por segmento: número de clientes y saldo promedio.

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 23.** Número de pedidos por año (`YEAR(o_orderdate)`), ordenado por año.

In [0]:
%sql
-- tu consulta aquí

## Día 6 · HAVING (filtrar grupos)
**Ejemplo resuelto:**

In [0]:
%sql
-- WHERE filtra FILAS antes de agrupar; HAVING filtra GRUPOS después de agrupar
SELECT o_orderpriority, COUNT(*) AS num_pedidos
FROM samples.tpch.orders
WHERE o_orderstatus = 'F'
GROUP BY o_orderpriority
HAVING COUNT(*) > 100000

**Ejercicio 24.** Clientes (`o_custkey`) con más de 30 pedidos, ordenados de más a menos pedidos.

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 25.** Segmentos cuyo saldo promedio sea mayor a 4500.

In [0]:
%sql
-- tu consulta aquí

**Ejercicio 26.** Clientes cuyo gasto total (suma de `o_totalprice`) supere 4,000,000. Muestra también cuántos pedidos tienen.

In [0]:
%sql
-- tu consulta aquí

## Día 7 · Repaso
**Ejercicio 27.** **Reto final (todo junto):** para los pedidos de 1996, por prioridad: número de pedidos, precio promedio (2 decimales) y pedido más caro. Muestra solo las prioridades con más de 45,000 pedidos, ordenadas por precio promedio de mayor a menor.

In [0]:
%sql
-- tu consulta aquí

# Parte B · PySpark: los mismos ejercicios con DataFrames
Hazlos el día que viste el tema en SQL. Así ves que es la **misma lógica con otra sintaxis**.

In [0]:
from pyspark.sql import functions as F
orders = spark.table("samples.tpch.orders")
customer = spark.table("samples.tpch.customer")

**PS1.** Ejercicio 2 con DataFrames: `select` de 3 columnas y `limit(20)`.

In [0]:
# tu código aquí

**PS2.** Ejercicio 8: `filter` con dos condiciones. Ojo: en PySpark se usa `&` (no `and`) y cada condición va entre paréntesis.

In [0]:
# tu código aquí

**PS3.** Ejercicio 13: los 10 pedidos más caros con `orderBy`.

In [0]:
# tu código aquí

**PS4.** Ejercicio 17: `agg` con `F.sum`, `F.avg`, `F.min`, `F.max`.

In [0]:
# tu código aquí

**PS5.** Ejercicio 21: `groupBy` + `agg` + `orderBy`.

In [0]:
# tu código aquí

**PS6.** Ejercicio 24 (HAVING): en PySpark, HAVING es simplemente un `filter` **después** del `groupBy`.

In [0]:
# tu código aquí

**PS7.** Reto final (27) con DataFrames.

In [0]:
# tu código aquí

# Parte C · Python puro (bloque de 3 h)
Aquí aprendes las estructuras de Python resolviendo **los mismos problemas que en SQL**, pero a mano. Primero ejecuta la celda con los datos.

In [0]:
# Tu "tabla" en Python puro: una lista de diccionarios (cada dict = una fila)
pedidos = [
    {"id": 1, "cliente": 101, "estado": "F", "prioridad": "1-URGENT", "total": 1520.50},
    {"id": 2, "cliente": 102, "estado": "O", "prioridad": "3-MEDIUM", "total": 830.00},
    {"id": 3, "cliente": 101, "estado": "F", "prioridad": "2-HIGH",   "total": 2310.75},
    {"id": 4, "cliente": 103, "estado": "P", "prioridad": "5-LOW",    "total": 120.00},
    {"id": 5, "cliente": 102, "estado": "F", "prioridad": "1-URGENT", "total": 4100.00},
    {"id": 6, "cliente": 104, "estado": "O", "prioridad": "2-HIGH",   "total": 990.90},
    {"id": 7, "cliente": 101, "estado": "O", "prioridad": "5-LOW",    "total": 60.25},
    {"id": 8, "cliente": 103, "estado": "F", "prioridad": "3-MEDIUM", "total": 1750.00},
]

**P1.** Día 1 · **Tipos.** Crea las variables `id_pedido` (int), `total` (float), `estado` (str) y `urgente` (bool). Imprime el tipo de cada una con `type()` y luego una frase con f-string: `Pedido 1 - total $1520.50`.

In [0]:
# tu código aquí

**P2.** Día 2 · **Listas + condicionales (= WHERE).** Con un `for` y un `if`, crea la lista `grandes` con los pedidos cuyo total sea mayor a 1000.

In [0]:
# tu código aquí

**P3.** Día 2 · **Función.** Escribe `filtrar_por_estado(filas, estado)` que devuelva solo los pedidos con ese estado. Pruébala con `"F"`.

In [0]:
# tu código aquí

**P4.** Día 3 · **Ordenar (= ORDER BY).** Usa `sorted` con `key=lambda` para obtener los 3 pedidos más caros.

In [0]:
# tu código aquí

**P5.** Día 4 · **Agregaciones.** Escribe `resumen(filas)` que devuelva un dict con `cantidad`, `suma`, `promedio`, `minimo` y `maximo` del campo `total`.

In [0]:
# tu código aquí

**P6.** Día 5 · **Comprensiones.** En una línea cada una: (a) lista de ids de pedidos urgentes (`prioridad == "1-URGENT"`), (b) set de clientes distintos, (c) dict `{id: total}`.

In [0]:
# tu código aquí

**P7.** Día 5 · **GROUP BY a mano.** Con un diccionario, cuenta cuántos pedidos hay por `estado` y suma el `total` por `cliente`.

In [0]:
# tu código aquí

**P8.** Día 6 · **HAVING a mano.** A partir de `suma_cliente`, quédate solo con los clientes que gastaron más de 2000 (usa una comprensión de diccionario).

In [0]:
# tu código aquí

**P9.** Día 7 · **Puente Spark → Python.** Trae 1000 pedidos reales de `samples.tpch.orders` con `.collect()`, conviértelos en diccionarios y aplica tu función `resumen`. Ojo: tu función usa la clave `total`, así que renombra `o_totalprice` en el `select` y convierte a `float` (Spark devuelve `Decimal`). Luego calcula lo mismo con Spark y compara.

In [0]:
# tu código aquí

## Cierre de la semana

- [ ] Hiciste los 27 ejercicios SQL, los 7 de PySpark y los 9 de Python.
- [ ] Comparaste tus respuestas con el notebook de soluciones **después** de intentarlo.
- [ ] Hiciste commit & push todos los días.
- [ ] Anotaste: ¿qué diferencia hay entre WHERE y HAVING? Explícalo en 2 líneas con tus palabras (te va a servir para una publicación).

**Para tu publicación de avance:** una captura de la misma consulta en SQL y en PySpark lado a lado, más esa explicación de WHERE vs. HAVING.